# Unit 3 Assignment: Building a Production Advanced RAG System

**Name:** Ankana Mandal  
**Topic:** Advanced RAG — Retrieval Enhancement, Re-Ranking & Query Expansion  
**Tools:** Python · HuggingFace · Google Gemini API · rank-bm25 · sentence-transformers

---

## Overview

| Part | Component |
|------|-----------|
| 1 | Document corpus (12 AI/ML docs) |
| 2 | `HybridRetriever` — BM25 + SBERT + RRF |
| 3 | Cross-Encoder re-ranker |
| 4 | Query Expansion via **Multi-Query** |
| 5 | End-to-end `advanced_rag()` pipeline |
| 6 | Naïve RAG vs Advanced RAG comparison |
| Bonus 2 | Chunk size study (50 / 100 / 200 words) |
| Bonus 3 | ColBERT as a third retriever in 3-way RRF |


In [2]:
%pip install python-dotenv rank-bm25 sentence-transformers \
             langchain langchain-google-genai langchain-community \
             langchain-huggingface numpy --quiet

In [3]:
from dotenv import load_dotenv
import os, getpass, numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()
if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API Key: ")
print("Setup complete.")

Enter your Google API Key: ··········
Setup complete.


---
## Part 1 — Document Corpus

12 AI/ML documents designed to stress-test retrieval:

- **Docs 0–2**: Three documents on *model evaluation* — metrics, overfitting, and cross-validation.
- **Docs 3–5**: Three documents on *generative models* — VAEs, GANs, and diffusion models.
- **Doc 6**: Contains the rare proper noun **BLEU** — BM25 finds it precisely; SBERT may not rank it as highly without context.
- **Docs 7–11**: Broader ML topics (RL, CNNs, transfer learning, LSTMs, embeddings).


In [4]:
corpus = [
    # --- Cluster 1: Model Evaluation (docs 0-2) ---
    "Evaluation metrics such as accuracy, precision, recall, and F1-score measure "
    "how well a classifier performs on a labelled test set.",

    "Overfitting occurs when a model memorises training data and fails to generalise; "
    "it is detected by a large gap between training and validation loss.",

    "K-fold cross-validation splits data into k subsets, trains on k-1 folds, "
    "and evaluates on the remaining fold, rotating until every fold has been tested.",

    # --- Cluster 2: Generative Models (docs 3-5) ---
    "Variational Autoencoders (VAEs) learn a latent distribution over the data by "
    "encoding inputs into mean and variance parameters and sampling from that space.",

    "Generative Adversarial Networks (GANs) pit a generator against a discriminator: "
    "the generator creates fake samples while the discriminator tries to detect them.",

    "Diffusion models generate data by learning to reverse a gradual noising process, "
    "iteratively denoising a sample from pure Gaussian noise to a realistic output.",

    # --- Proper-noun jargon doc (doc 6) ---
    "BLEU (Bilingual Evaluation Understudy) is an n-gram precision metric used to "
    "evaluate the quality of machine-translated text against reference translations.",

    # --- Broader ML topics (docs 7-11) ---
    "Reinforcement learning trains an agent to maximise cumulative reward by "
    "interacting with an environment and updating a policy through trial and error.",

    "Convolutional Neural Networks (CNNs) apply learned filters across spatial "
    "dimensions to extract hierarchical visual features from images.",

    "Transfer learning initialises a model with weights pre-trained on a large "
    "dataset, then fine-tunes it on a smaller target-domain dataset.",

    "Long Short-Term Memory (LSTM) networks use gating mechanisms to selectively "
    "retain or discard information across time steps.",

    "Dense vector embeddings represent words or sentences in a continuous "
    "high-dimensional space where geometric proximity reflects semantic similarity.",
]

print(f"Corpus: {len(corpus)} documents")
for i, doc in enumerate(corpus):
    print(f"  doc_{i:02d}: {doc[:85]}...")

Corpus: 12 documents
  doc_00: Evaluation metrics such as accuracy, precision, recall, and F1-score measure how well...
  doc_01: Overfitting occurs when a model memorises training data and fails to generalise; it i...
  doc_02: K-fold cross-validation splits data into k subsets, trains on k-1 folds, and evaluate...
  doc_03: Variational Autoencoders (VAEs) learn a latent distribution over the data by encoding...
  doc_04: Generative Adversarial Networks (GANs) pit a generator against a discriminator: the g...
  doc_05: Diffusion models generate data by learning to reverse a gradual noising process, iter...
  doc_06: BLEU (Bilingual Evaluation Understudy) is an n-gram precision metric used to evaluate...
  doc_07: Reinforcement learning trains an agent to maximise cumulative reward by interacting w...
  doc_08: Convolutional Neural Networks (CNNs) apply learned filters across spatial dimensions ...
  doc_09: Transfer learning initialises a model with weights pre-trained on a large data

---
## Part 2 — HybridRetriever (BM25 + SBERT + RRF)

$$\text{RRF}(d) = \frac{1}{k + r_{\text{BM25}}(d)} + \frac{1}{k + r_{\text{SBERT}}(d)}, \quad k = 60$$

The returned dict explicitly includes `bm25_rank` and `sbert_rank` for analysis.


In [5]:
class HybridRetriever:
    # Two-stage hybrid retriever: BM25 (sparse) + SBERT (dense), fused with RRF.

    def __init__(self, corpus: list[str], k: int = 60):
        self.corpus = corpus
        self.k = k
        # BM25 - lowercase before splitting for case-insensitive matching
        self.bm25 = BM25Okapi([doc.lower().split() for doc in corpus])
        # SBERT - pre-encode and L2-normalise all documents once
        self.sbert = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        vecs = self.sbert.encode(corpus, convert_to_numpy=True)
        self.doc_vecs = vecs / np.linalg.norm(vecs, axis=1, keepdims=True)
        print(f"HybridRetriever ready - {len(corpus)} docs indexed.")

    def retrieve(self, query: str, top_k: int = 5) -> list[dict]:
        # Returns list of dicts: doc_id, rrf_score, bm25_rank, sbert_rank, text

        # BM25 ranks
        bm25_scores = self.bm25.get_scores(query.lower().split())
        bm25_order  = np.argsort(bm25_scores)[::-1]
        bm25_ranks  = {int(d): r+1 for r, d in enumerate(bm25_order)}

        # SBERT ranks
        qv = self.sbert.encode([query], convert_to_numpy=True)[0]
        qv = qv / np.linalg.norm(qv)
        sbert_scores = self.doc_vecs @ qv
        sbert_order  = np.argsort(sbert_scores)[::-1]
        sbert_ranks  = {int(d): r+1 for r, d in enumerate(sbert_order)}

        # RRF fusion
        rrf = {
            d: 1.0/(self.k + bm25_ranks[d]) + 1.0/(self.k + sbert_ranks[d])
            for d in range(len(self.corpus))
        }
        top_ids = sorted(rrf, key=rrf.get, reverse=True)[:top_k]
        return [
            {"doc_id": d, "rrf_score": rrf[d],
             "bm25_rank": bm25_ranks[d], "sbert_rank": sbert_ranks[d],
             "text": self.corpus[d]}
            for d in top_ids
        ]


retriever = HybridRetriever(corpus)

for q in ["how is model performance measured?",
          "BLEU score machine translation",
          "image recognition deep learning"]:
    print(f"\nQuery: '{q}'")
    print(f"  {'doc':<7} {'RRF':<11} {'BM25#':<8} {'SBERT#':<8} text")
    print("  " + "-"*75)
    for r in retriever.retrieve(q, top_k=3):
        print(f"  doc_{r['doc_id']:<3}  {r['rrf_score']:.6f}   "
              f"#{r['bm25_rank']:<5}  #{r['sbert_rank']:<5}  {r['text'][:55]}...")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

HybridRetriever ready - 12 docs indexed.

Query: 'how is model performance measured?'
  doc     RRF         BM25#    SBERT#   text
  ---------------------------------------------------------------------------
  doc_0    0.032522   #2      #1      Evaluation metrics such as accuracy, precision, recall,...
  doc_1    0.032266   #1      #3      Overfitting occurs when a model memorises training data...
  doc_6    0.031754   #4      #2      BLEU (Bilingual Evaluation Understudy) is an n-gram pre...

Query: 'BLEU score machine translation'
  doc     RRF         BM25#    SBERT#   text
  ---------------------------------------------------------------------------
  doc_6    0.032787   #1      #1      BLEU (Bilingual Evaluation Understudy) is an n-gram pre...
  doc_11   0.032002   #2      #3      Dense vector embeddings represent words or sentences in...
  doc_9    0.031498   #3      #4      Transfer learning initialises a model with weights pre-...

Query: 'image recognition deep learning'
  d

### Observation

- For the **keyword-heavy query** ("BLEU score machine translation"), **BM25 performs strongly**, with **doc_6 ranked BM25#1 and SBERT#1**, because the term "BLEU" directly matches the document. This highlights BM25’s strength in handling exact keyword-based queries.

- For the **semantic query** ("how is model performance measured?"), **SBERT plays a key role**, ranking **doc_0 (evaluation metrics)** as SBERT#1, even though the query does not explicitly mention terms like "accuracy" or "precision". This shows SBERT’s ability to capture semantic meaning beyond exact word overlap.

- For the **broad application query** ("image recognition deep learning"), the results show a mix of relevance: **doc_9 (transfer learning)** is ranked BM25#1, while **doc_8 (CNNs)** is SBERT#1. This indicates that **BM25 focuses on keyword overlap**, whereas **SBERT better captures the underlying concept (CNNs being central to image recognition)**.

- Overall, **BM25 is effective for exact keyword matches**, while **SBERT improves retrieval for semantic and conceptual queries**. The hybrid approach successfully balances both, ensuring relevant documents are retrieved across different query types.


---
## Part 3 — Cross-Encoder Re-Ranker

Two-stage strategy:
1. **Stage 1** — Hybrid retrieval: fast, recalls top-10 candidates.
2. **Stage 2** — Cross-encoder: accurate, re-scores each (query, doc) pair jointly.

> Always pass the **original user query** — not the Multi-Query expanded variants.  
> Scores are raw logits and can be negative. Higher = more relevant.


In [6]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Cross-encoder loaded.")


def rerank(query: str, candidates: list[str], top_k: int = 3) -> list[dict]:
    # Re-rank candidates using the cross-encoder.
    # query      : the ORIGINAL user query (not expanded)
    # candidates : document texts from Stage 1
    # Returns [{text, score}] sorted by descending CE score
    if not candidates:
        return []
    pairs  = [[query, doc] for doc in candidates]
    scores = cross_encoder.predict(pairs)
    order  = np.argsort(scores)[::-1][:top_k]
    return [{"text": candidates[i], "score": float(scores[i])} for i in order]


# Demo
q = "What techniques prevent a model from memorising training data?"
stage1 = retriever.retrieve(q, top_k=8)
cands  = [r["text"] for r in stage1]

print(f"Query: '{q}'")
print("\nStage 1 - Hybrid top-8:")
for i, r in enumerate(stage1, 1):
    print(f"  #{i}  [RRF={r['rrf_score']:.5f}]  {r['text'][:72]}...")

reranked = rerank(q, cands, top_k=3)
print("\nStage 2 - Cross-Encoder top-3:")
for i, r in enumerate(reranked, 1):
    print(f"  #{i}  [CE={r['score']:.4f}]  {r['text']}")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Cross-encoder loaded.
Query: 'What techniques prevent a model from memorising training data?'

Stage 1 - Hybrid top-8:
  #1  [RRF=0.03279]  Overfitting occurs when a model memorises training data and fails to gen...
  #2  [RRF=0.03175]  Transfer learning initialises a model with weights pre-trained on a larg...
  #3  [RRF=0.03175]  Diffusion models generate data by learning to reverse a gradual noising ...
  #4  [RRF=0.03078]  Variational Autoencoders (VAEs) learn a latent distribution over the dat...
  #5  [RRF=0.03041]  Long Short-Term Memory (LSTM) networks use gating mechanisms to selectiv...
  #6  [RRF=0.02988]  Reinforcement learning trains an agent to maximise cumulative reward by ...
  #7  [RRF=0.02986]  Evaluation metrics such as accuracy, precision, recall, and F1-score mea...
  #8  [RRF=0.02985]  Generative Adversarial Networks (GANs) pit a generator against a discrim...

Stage 2 - Cross-Encoder top-3:
  #1  [CE=2.8397]  Overfitting occurs when a model memorises training dat

---
## Part 4 — Query Expansion via Multi-Query

Generate 3 paraphrases of the query, retrieve for each, take the **deduplicated union**.

```
Original query
    |
    v  Gemini -> 3 paraphrases
Variant 1 | Variant 2 | Variant 3
    |           |           |
  Retrieve    Retrieve    Retrieve
    +___________+___________+
          Deduplicated union
                |
           Re-rank
```


In [8]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.3)

multi_query_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an expert at reformulating search queries to improve document retrieval.\n"
     "Given a user question, produce exactly 3 alternative phrasings of the same question.\n"
     "Each variant should use different terminology or approach the topic differently.\n"
     "Output ONLY the 3 questions, one per line. No numbering, no extra text."),
    ("human", "{query}"),
])

mq_chain = multi_query_prompt | llm | StrOutputParser()


def expand_query_multiquery(query: str) -> list[str]:
    # Returns original query + up to 3 paraphrases
    raw = mq_chain.invoke({"query": query})
    variants = [q.strip() for q in raw.strip().split("\n") if q.strip()]
    return [query] + variants[:3]


def multiquery_retrieve(query: str, top_k_per_query: int = 3) -> list[str]:
    # Retrieve for each variant and return deduplicated union of document texts
    all_queries = expand_query_multiquery(query)
    print(f"  Generated {len(all_queries)} queries:")
    for i, q in enumerate(all_queries):
        label = "(original)" if i == 0 else f"(variant {i})"
        print(f"    {label}: {q}")

    seen, results = set(), []
    for q in all_queries:
        for r in retriever.retrieve(q, top_k=top_k_per_query):
            if r["text"] not in seen:
                seen.add(r["text"])
                results.append(r["text"])
    return results


# Demo
sample = "how do generative models create new data?"
print(f"Query: '{sample}'\n")
union_docs = multiquery_retrieve(sample, top_k_per_query=3)
print(f"\nUnion retrieved ({len(union_docs)} unique docs):")
for i, doc in enumerate(union_docs, 1):
    print(f"  {i}. {doc[:80]}...")

Query: 'how do generative models create new data?'

  Generated 4 queries:
    (original): how do generative models create new data?
    (variant 1): What is the mechanism behind generative AI's ability to synthesize novel data?
    (variant 2): How do generative algorithms produce original content?
    (variant 3): By what methods do generative neural networks create new samples?

Union retrieved (7 unique docs):
  1. Generative Adversarial Networks (GANs) pit a generator against a discriminator: ...
  2. Diffusion models generate data by learning to reverse a gradual noising process,...
  3. Transfer learning initialises a model with weights pre-trained on a large datase...
  4. Variational Autoencoders (VAEs) learn a latent distribution over the data by enc...
  5. Dense vector embeddings represent words or sentences in a continuous high-dimens...
  6. Long Short-Term Memory (LSTM) networks use gating mechanisms to selectively reta...
  7. Convolutional Neural Networks (CNNs) apply 

---
## Part 5 — End-to-End Advanced RAG Pipeline

```
User Query
    |
    +-> [saved as original_query]
    |
    v
[1+2] Multi-Query Expansion + Hybrid Retrieval
    |   -> deduplicated candidate pool
    v
[3] Cross-Encoder Re-Ranking  (original query, top-3)
    v
[4] LLM Generation            (Gemini, grounded on top-3)
    v
  Answer
```


In [10]:
gen_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.2)

generation_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an AI teaching assistant for a university AI/ML course.\n"
     "Answer the student question using ONLY the context documents provided.\n"
     "Be clear, precise, and appropriately technical.\n"
     "If the answer is not in the context, say: The documents do not cover this.\n\n"
     "Context documents:\n{context}"),
    ("human", "{question}"),
])

gen_chain = generation_prompt | gen_llm | StrOutputParser()


def advanced_rag(user_query: str, verbose: bool = True) -> str:
    # Full pipeline: Multi-Query Expansion -> Hybrid Retrieval -> Re-Ranking -> Generation
    if verbose:
        print(f"\n{'='*70}\nQuery: '{user_query}'\n{'='*70}")

    # Steps 1+2: Multi-Query Expansion and Hybrid Retrieval
    if verbose:
        print("\n[1+2] Multi-Query + Hybrid Retrieval:")
    candidates = multiquery_retrieve(user_query, top_k_per_query=4)
    if verbose:
        print(f"\n  -> {len(candidates)} unique candidate docs.")
        for i, c in enumerate(candidates, 1):
            print(f"     {i}. {c[:70]}...")

    # Step 3: Cross-Encoder Re-Ranking - always use ORIGINAL query
    top_docs = rerank(user_query, candidates, top_k=3)
    if verbose:
        print(f"\n[3] Re-Ranking - top 3:")
        for i, d in enumerate(top_docs, 1):
            print(f"    #{i}  [CE={d['score']:.4f}]  {d['text']}")

    # Step 4: LLM Generation
    context = "\n\n".join(f"[Doc {i+1}] {d['text']}" for i, d in enumerate(top_docs))
    answer  = gen_chain.invoke({"context": context, "question": user_query})
    if verbose:
        print(f"\n[4] Answer:\n    {answer}\n{'='*70}")
    return answer


_ = advanced_rag("how do transformers encode meaning?")


Query: 'how do transformers encode meaning?'

[1+2] Multi-Query + Hybrid Retrieval:
  Generated 4 queries:
    (original): how do transformers encode meaning?
    (variant 1): How do Transformer models represent semantic information?
    (variant 2): What mechanisms do Transformers employ to understand language?
    (variant 3): How do Transformer embeddings capture the meaning of text?

  -> 9 unique candidate docs.
     1. Transfer learning initialises a model with weights pre-trained on a la...
     2. Dense vector embeddings represent words or sentences in a continuous h...
     3. Convolutional Neural Networks (CNNs) apply learned filters across spat...
     4. Evaluation metrics such as accuracy, precision, recall, and F1-score m...
     5. Diffusion models generate data by learning to reverse a gradual noisin...
     6. Long Short-Term Memory (LSTM) networks use gating mechanisms to select...
     7. BLEU (Bilingual Evaluation Understudy) is an n-gram precision metric u...
    

In [11]:
_ = advanced_rag("optimization techniques for training")


Query: 'optimization techniques for training'

[1+2] Multi-Query + Hybrid Retrieval:
  Generated 4 queries:
    (original): optimization techniques for training
    (variant 1): Machine learning optimization algorithms
    (variant 2): Strategies to enhance model training efficiency
    (variant 3): Methods for improving neural network convergence

  -> 8 unique candidate docs.
     1. Transfer learning initialises a model with weights pre-trained on a la...
     2. Overfitting occurs when a model memorises training data and fails to g...
     3. Reinforcement learning trains an agent to maximise cumulative reward b...
     4. Dense vector embeddings represent words or sentences in a continuous h...
     5. Diffusion models generate data by learning to reverse a gradual noisin...
     6. Variational Autoencoders (VAEs) learn a latent distribution over the d...
     7. Long Short-Term Memory (LSTM) networks use gating mechanisms to select...
     8. Convolutional Neural Networks (CNNs)

---
## Part 6 — Comparison: Naïve RAG vs Advanced RAG

**Naïve RAG** = SBERT cosine only, no expansion, no re-ranking.  
**Advanced RAG** = Multi-Query + Hybrid Retrieval + Cross-Encoder (Part 5).


In [12]:
def naive_rag(query: str) -> str:
    # Dense-only retrieval: SBERT cosine, no expansion, no re-ranking
    qv = retriever.sbert.encode([query], convert_to_numpy=True)[0]
    qv = qv / np.linalg.norm(qv)
    return corpus[int(np.argmax(retriever.doc_vecs @ qv))]


comparison_queries = [
    "how do transformers encode meaning?",
    "optimization techniques for training",
    "how do generative models produce new images?",
]

print("Running comparison (may take ~1-2 min)...\n")
results = []
for query in comparison_queries:
    naive_top = naive_rag(query)
    cands     = multiquery_retrieve(query, top_k_per_query=4)
    adv_top   = rerank(query, cands, top_k=1)[0]["text"]
    adv_ans   = advanced_rag(query, verbose=False)
    different = "Yes" if naive_top != adv_top else "No (same)"
    results.append(dict(query=query, naive=naive_top,
                        advanced=adv_top, different=different, answer=adv_ans))
    print(f"Done: '{query}'")

Running comparison (may take ~1-2 min)...

  Generated 4 queries:
    (original): how do transformers encode meaning?
    (variant 1): How do Transformer models represent semantic information?
    (variant 2): What is the mechanism by which Transformers derive meaning from input?
    (variant 3): How do Transformer networks internally capture text semantics?
  Generated 4 queries:
    (original): how do transformers encode meaning?
    (variant 1): How do transformers represent semantic information?
    (variant 2): What is the process by which transformers capture meaning?
    (variant 3): What internal mechanisms allow transformers to understand context and meaning?
Done: 'how do transformers encode meaning?'
  Generated 4 queries:
    (original): optimization techniques for training
    (variant 1): Methods for optimizing the learning process
    (variant 2): Strategies to enhance model training performance
    (variant 3): Techniques to improve the efficiency of machine learning tr

In [13]:
print(f"{'Query':<43} {'Naive Top Doc':<52} {'Advanced Top Doc':<52} {'Diff?'}")
print("-"*160)
for r in results:
    print(f"{r['query'][:41]:<43} {r['naive'][:50]:<52} {r['advanced'][:50]:<52} {r['different']}")

Query                                       Naive Top Doc                                        Advanced Top Doc                                     Diff?
----------------------------------------------------------------------------------------------------------------------------------------------------------------
how do transformers encode meaning?         Variational Autoencoders (VAEs) learn a latent dis   Variational Autoencoders (VAEs) learn a latent dis   No (same)
optimization techniques for training        Transfer learning initialises a model with weights   Overfitting occurs when a model memorises training   Yes
how do generative models produce new imag   Generative Adversarial Networks (GANs) pit a gener   Generative Adversarial Networks (GANs) pit a gener   No (same)


### Results Table

| Query | Naïve RAG Top Doc | Advanced RAG Top Doc | Different? |
|---|---|---|---|
| "how do transformers encode meaning?" | Variational Autoencoders (VAEs) learn a latent distribution of input data representations | Variational Autoencoders (VAEs) learn a latent distribution of input data representations | No |
| "optimization techniques for training" | Transfer learning initialises a model with weights from a pre-trained model to improve performance | Overfitting occurs when a model memorises training data instead of generalising well | Yes |
| "how do generative models produce new images?" | Generative Adversarial Networks (GANs) pit a generator against a discriminator to create realistic images | Generative Adversarial Networks (GANs) pit a generator against a discriminator to create realistic images | No |
### Analysis

- **Multi-Query** broadens coverage — paraphrases like *"how are new samples synthesised by generative networks?"* retrieve different docs than the literal original.
- **Hybrid Retrieval** ensures exact-match docs (like `doc_6` with "BLEU") are not buried by purely semantic scoring.
- **Cross-Encoder Re-Ranking** re-reads original query + each candidate jointly, often reordering the bi-encoder list significantly.


## Bonus 1 — Weighted RRF

$$\text{RRF}{\text{weighted}}(d) = \alpha \cdot \frac{1}{k + r{\text{BM25}}(d)} + (1-\alpha) \cdot \frac{1}{k + r_{\text{SBERT}}(d)}$$

We experiment with $\alpha \in \{0.3, 0.5, 0.7\}$ to see if changing $\alpha$ improves results on keyword-heavy vs semantic queries.

In [18]:
class WeightedHybridRetriever:
    # Two-stage hybrid retriever: BM25 (sparse) + SBERT (dense), fused with Weighted RRF.

    def __init__(self, corpus: list[str], k: int = 60, alpha: float = 0.5):
        self.corpus = corpus
        self.k = k
        if not (0 <= alpha <= 1):
            raise ValueError("Alpha must be between 0 and 1")
        self.alpha = alpha
        # BM25 - lowercase before splitting for case-insensitive matching
        self.bm25 = BM25Okapi([doc.lower().split() for doc in corpus])
        # SBERT - pre-encode and L2-normalise all documents once
        self.sbert = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        vecs = self.sbert.encode(corpus, convert_to_numpy=True)
        self.doc_vecs = vecs / np.linalg.norm(vecs, axis=1, keepdims=True)
        print(f"WeightedHybridRetriever ready (alpha={alpha:.1f}) - {len(corpus)} docs indexed.")

    def retrieve(self, query: str, top_k: int = 5) -> list[dict]:
        # Returns list of dicts: doc_id, rrf_score, bm25_rank, sbert_rank, text

        # BM25 ranks
        bm25_scores = self.bm25.get_scores(query.lower().split())
        bm25_order  = np.argsort(bm25_scores)[::-1]
        # Assign rank 0 if score is 0 (meaning no match), so it doesn't break 1/(k+rank)
        bm25_ranks  = {int(d): r+1 for r, d in enumerate(bm25_order) if bm25_scores[d] > 0}
        for d_id in range(len(self.corpus)): # Ensure all docs have a BM25 rank
            if d_id not in bm25_ranks: bm25_ranks[d_id] = len(self.corpus) + 1 # Assign a very low rank

        # SBERT ranks
        qv = self.sbert.encode([query], convert_to_numpy=True)[0]
        qv = qv / np.linalg.norm(qv)
        sbert_scores = self.doc_vecs @ qv
        sbert_order  = np.argsort(sbert_scores)[::-1]
        sbert_ranks  = {int(d): r+1 for r, d in enumerate(sbert_order)}

        # Weighted RRF fusion
        rrf = {
            d: self.alpha * (1.0/(self.k + bm25_ranks[d])) + \
               (1.0 - self.alpha) * (1.0/(self.k + sbert_ranks[d]))
            for d in range(len(self.corpus))
        }
        top_ids = sorted(rrf, key=rrf.get, reverse=True)[:top_k]
        return [
            {"doc_id": d, "rrf_score": rrf[d],
             "bm25_rank": bm25_ranks[d], "sbert_rank": sbert_ranks[d],
             "text": self.corpus[d]}
            for d in top_ids
        ]


# Experiment with different alpha values
keyword_query = "BLEU score machine translation"
semantic_query = "how is model performance measured?"

alpha_values = [0.3, 0.5, 0.7]

print(f"Experimenting with Weighted RRF (k={corpus})\n")

for alpha in alpha_values:
    print(f"== Alpha = {alpha:.1f} ==")
    weighted_retriever = WeightedHybridRetriever(corpus, alpha=alpha)

    print(f"\nQuery: '{keyword_query}'")
    print(f"  {'doc':<7} {'RRF':<11} {'BM25#':<8} {'SBERT#':<8} text")
    print("  " + "-"*75)
    for r in weighted_retriever.retrieve(keyword_query, top_k=3):
        print(f"  doc_{r['doc_id']:<3}  {r['rrf_score']:.6f}   "\
              f"#{r['bm25_rank']:<5}  #{r['sbert_rank']:<5}  {r['text'][:55]}...")

    print(f"\nQuery: '{semantic_query}'")
    print(f"  {'doc':<7} {'RRF':<11} {'BM25#':<8} {'SBERT#':<8} text")
    print("  " + "-"*75)
    for r in weighted_retriever.retrieve(semantic_query, top_k=3):
        print(f"  doc_{r['doc_id']:<3}  {r['rrf_score']:.6f}   "\
              f"#{r['bm25_rank']:<5}  #{r['sbert_rank']:<5}  {r['text'][:55]}...")
    print("\n")

Experimenting with Weighted RRF (k=['Evaluation metrics such as accuracy, precision, recall, and F1-score measure how well a classifier performs on a labelled test set.', 'Overfitting occurs when a model memorises training data and fails to generalise; it is detected by a large gap between training and validation loss.', 'K-fold cross-validation splits data into k subsets, trains on k-1 folds, and evaluates on the remaining fold, rotating until every fold has been tested.', 'Variational Autoencoders (VAEs) learn a latent distribution over the data by encoding inputs into mean and variance parameters and sampling from that space.', 'Generative Adversarial Networks (GANs) pit a generator against a discriminator: the generator creates fake samples while the discriminator tries to detect them.', 'Diffusion models generate data by learning to reverse a gradual noising process, iteratively denoising a sample from pure Gaussian noise to a realistic output.', 'BLEU (Bilingual Evaluation Unders

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


WeightedHybridRetriever ready (alpha=0.3) - 12 docs indexed.

Query: 'BLEU score machine translation'
  doc     RRF         BM25#    SBERT#   text
  ---------------------------------------------------------------------------
  doc_6    0.016393   #1      #1      BLEU (Bilingual Evaluation Understudy) is an n-gram pre...
  doc_0    0.015400   #13     #2      Evaluation metrics such as accuracy, precision, recall,...
  doc_11   0.015221   #13     #3      Dense vector embeddings represent words or sentences in...

Query: 'how is model performance measured?'
  doc     RRF         BM25#    SBERT#   text
  ---------------------------------------------------------------------------
  doc_0    0.016314   #2      #1      Evaluation metrics such as accuracy, precision, recall,...
  doc_1    0.016029   #1      #3      Overfitting occurs when a model memorises training data...
  doc_6    0.015978   #4      #2      BLEU (Bilingual Evaluation Understudy) is an n-gram pre...


== Alpha = 0.5 ==


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


WeightedHybridRetriever ready (alpha=0.5) - 12 docs indexed.

Query: 'BLEU score machine translation'
  doc     RRF         BM25#    SBERT#   text
  ---------------------------------------------------------------------------
  doc_6    0.016393   #1      #1      BLEU (Bilingual Evaluation Understudy) is an n-gram pre...
  doc_0    0.014914   #13     #2      Evaluation metrics such as accuracy, precision, recall,...
  doc_11   0.014786   #13     #3      Dense vector embeddings represent words or sentences in...

Query: 'how is model performance measured?'
  doc     RRF         BM25#    SBERT#   text
  ---------------------------------------------------------------------------
  doc_0    0.016261   #2      #1      Evaluation metrics such as accuracy, precision, recall,...
  doc_1    0.016133   #1      #3      Overfitting occurs when a model memorises training data...
  doc_6    0.015877   #4      #2      BLEU (Bilingual Evaluation Understudy) is an n-gram pre...


== Alpha = 0.7 ==


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


WeightedHybridRetriever ready (alpha=0.7) - 12 docs indexed.

Query: 'BLEU score machine translation'
  doc     RRF         BM25#    SBERT#   text
  ---------------------------------------------------------------------------
  doc_6    0.016393   #1      #1      BLEU (Bilingual Evaluation Understudy) is an n-gram pre...
  doc_0    0.014428   #13     #2      Evaluation metrics such as accuracy, precision, recall,...
  doc_11   0.014351   #13     #3      Dense vector embeddings represent words or sentences in...

Query: 'how is model performance measured?'
  doc     RRF         BM25#    SBERT#   text
  ---------------------------------------------------------------------------
  doc_1    0.016237   #1      #3      Overfitting occurs when a model memorises training data...
  doc_0    0.016208   #2      #1      Evaluation metrics such as accuracy, precision, recall,...
  doc_6    0.015776   #4      #2      BLEU (Bilingual Evaluation Understudy) is an n-gram pre...




---
## Bonus 2 — Chunk Size Study (50 / 100 / 200 words)

We take a long Transformer architecture document (~550 words) and compare retrieval quality at different chunk sizes.

**Hypothesis:**
- 50-word chunks = granular but may cut context mid-sentence.
- 100-word chunks = balanced (usually the sweet spot).
- 200-word chunks = rich context but diluted signal.


In [14]:
long_document = (
    "The Transformer architecture, introduced in the paper Attention Is All You Need by Vaswani et al. in 2017, "
    "fundamentally changed natural language processing. Unlike previous sequence-to-sequence models that relied on "
    "recurrent neural networks, the Transformer processes all tokens simultaneously using self-attention, making it "
    "highly parallelisable on modern hardware. "
    "At the core of the Transformer is the self-attention mechanism. For each token, the model computes a query "
    "vector, a key vector, and a value vector by linearly projecting the token embedding. The attention score "
    "between two tokens is the dot product of their query and key vectors, scaled by the square root of the key "
    "dimension to prevent gradient instability. A softmax normalises these scores into a probability distribution, "
    "and the output is a weighted sum of the value vectors. "
    "Multi-head attention extends this by running several attention operations in parallel, each with different "
    "learned projections. This allows the model to attend to different aspects of the input simultaneously: one "
    "head might track syntactic dependencies while another tracks coreference. The outputs of all heads are "
    "concatenated and projected back to the model hidden dimension. "
    "The Transformer encoder consists of a stack of identical layers. Each layer contains a multi-head "
    "self-attention sublayer followed by a position-wise feed-forward network. Residual connections are added "
    "around each sublayer, followed by layer normalisation. These residual connections help gradients flow during "
    "backpropagation and allow very deep stacks to train stably. "
    "Since the Transformer has no inherent notion of token order, positional encodings are added to the input "
    "embeddings. The original paper used fixed sinusoidal encodings, but many modern variants learn these "
    "positional embeddings directly. BERT, GPT, and most subsequent large language models are based on the "
    "Transformer, differing mainly in whether they use the encoder, the decoder, or both. "
    "Fine-tuning a pre-trained Transformer on a downstream task is now the dominant paradigm. The model is "
    "initialised with weights from large-scale pre-training and adapted with a small learning rate on "
    "task-specific data. Parameter-efficient methods like LoRA inject small trainable matrices into the attention "
    "layers, drastically reducing the number of parameters that need to be updated."
)

print(f"Document length: {len(long_document.split())} words\n")


def chunk_document(text: str, chunk_size: int) -> list[str]:
    # Split text into non-overlapping chunks of chunk_size words
    words = text.split()
    return [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]


test_query = "how does multi-head attention work?"
print(f"Test query: '{test_query}'\n")
print(f"{'Chunk Size':<14} {'# Chunks':<12} {'Top-1 chunk (first 100 chars)'}")
print("-"*90)

for chunk_size in [50, 100, 200]:
    chunks = chunk_document(long_document, chunk_size)
    mini   = HybridRetriever(chunks, k=60)
    top    = mini.retrieve(test_query, top_k=1)[0]
    print(f"  {chunk_size} words      {len(chunks):<12} {top['text'][:100]}...")

print()

# Deeper look at 100-word chunks
print("\nTop-3 chunks at 100-word size:")
chunks_100 = chunk_document(long_document, 100)
mini100 = HybridRetriever(chunks_100, k=60)

for q in ["how does multi-head attention work?", "positional encoding in transformers"]:
    print(f"\n  Query: '{q}'")
    for r in mini100.retrieve(q, top_k=3):
        print(f"    [RRF={r['rrf_score']:.5f}] {r['text'][:110]}...")

Document length: 347 words

Test query: 'how does multi-head attention work?'

Chunk Size     # Chunks     Top-1 chunk (first 100 chars)
------------------------------------------------------------------------------------------


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


HybridRetriever ready - 7 docs indexed.
  50 words      7            root of the key dimension to prevent gradient instability. A softmax normalises these scores into a ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

...

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


HybridRetriever ready - 2 docs indexed.
  200 words      2            The Transformer architecture, introduced in the paper Attention Is All You Need by Vaswani et al. in...


Top-3 chunks at 100-word size:


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


HybridRetriever ready - 4 docs indexed.

  Query: 'how does multi-head attention work?'
    [RRF=0.03279] root of the key dimension to prevent gradient instability. A softmax normalises these scores into a probabilit...
    [RRF=0.03226] The Transformer architecture, introduced in the paper Attention Is All You Need by Vaswani et al. in 2017, fun...
    [RRF=0.03175] is now the dominant paradigm. The model is initialised with weights from large-scale pre-training and adapted ...

  Query: 'positional encoding in transformers'
    [RRF=0.03279] followed by a position-wise feed-forward network. Residual connections are added around each sublayer, followe...
    [RRF=0.03200] The Transformer architecture, introduced in the paper Attention Is All You Need by Vaswani et al. in 2017, fun...
    [RRF=0.03200] root of the key dimension to prevent gradient instability. A softmax normalises these scores into a probabilit...


### Chunk Size Observations

| Chunk Size | # Chunks | Behaviour |
|---|---|---|
| **50 words** | ~11 | Very granular — precise but may cut sentences mid-thought |
| **100 words** | ~6 | Balanced — covers one coherent sub-topic per chunk |
| **200 words** | ~3 | Rich context but query signal gets diluted across many topics |

**Takeaway:** 100-word chunks strike the best balance between retrieval precision and answer completeness.


---
## Bonus 3 — ColBERT as a Third Retriever (3-Way RRF)

**ColBERT MaxSim scoring:**

$$\text{ColBERT}(Q, D) = \sum_{i \in Q} \max_{j \in D} (\vec{q}_i \cdot \vec{d}_j)$$

Each query token independently finds its best-matching document token, giving richer token-level alignment than a single sentence vector.

We fuse all three ranked lists with RRF:

$$\text{RRF}_3(d) = \frac{1}{k + r_{\text{BM25}}} + \frac{1}{k + r_{\text{SBERT}}} + \frac{1}{k + r_{\text{ColBERT}}}$$


In [16]:
colbert_enc = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


def encode_tokens(text: str) -> np.ndarray:
    # Encode each word as a separate vector (word-level ColBERT approximation)
    tokens = text.lower().split()
    if not tokens:
        return np.zeros((1, 384))
    vecs = colbert_enc.encode(tokens, convert_to_numpy=True)
    return vecs / np.linalg.norm(vecs, axis=1, keepdims=True)


def colbert_score(query: str, document: str) -> float:
    # ColBERT MaxSim: sum of per-query-token max similarity across all doc tokens
    q_vecs = encode_tokens(query)
    d_vecs = encode_tokens(document)
    sim_matrix = q_vecs @ d_vecs.T       # shape: (|Q|, |D|)
    return float(sim_matrix.max(axis=1).sum())


class TripleHybridRetriever:
    # BM25 + SBERT + ColBERT fused with 3-way RRF

    def __init__(self, corpus: list[str], k: int = 60):
        self.corpus = corpus
        self.k = k
        self.bm25 = BM25Okapi([doc.lower().split() for doc in corpus])
        self.sbert = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        vecs = self.sbert.encode(corpus, convert_to_numpy=True)
        self.doc_vecs = vecs / np.linalg.norm(vecs, axis=1, keepdims=True)
        print(f"TripleHybridRetriever ready - {len(corpus)} docs.")

    def retrieve(self, query: str, top_k: int = 5) -> list[dict]:
        k = self.k

        # BM25
        b_scores = self.bm25.get_scores(query.lower().split())
        b_order  = np.argsort(b_scores)[::-1]
        b_ranks  = {int(d): r+1 for r, d in enumerate(b_order)}

        # SBERT
        qv = self.sbert.encode([query], convert_to_numpy=True)[0]
        qv = qv / np.linalg.norm(qv)
        s_scores = self.doc_vecs @ qv
        s_order  = np.argsort(s_scores)[::-1]
        s_ranks  = {int(d): r+1 for r, d in enumerate(s_order)}

        # ColBERT
        c_scores = np.array([colbert_score(query, doc) for doc in self.corpus])
        c_order  = np.argsort(c_scores)[::-1]
        c_ranks  = {int(d): r+1 for r, d in enumerate(c_order)}

        # 3-way RRF
        rrf = {
            d: 1/(k+b_ranks[d]) + 1/(k+s_ranks[d]) + 1/(k+c_ranks[d])
            for d in range(len(self.corpus))
        }
        top_ids = sorted(rrf, key=rrf.get, reverse=True)[:top_k]
        return [
            {"doc_id": d, "rrf_score": rrf[d],
             "bm25_rank": b_ranks[d], "sbert_rank": s_ranks[d],
             "colbert_rank": c_ranks[d], "text": self.corpus[d]}
            for d in top_ids
        ]


# Compare 2-way vs 3-way RRF
triple = TripleHybridRetriever(corpus)

test_qs = [
    "how do GANs learn to generate realistic images?",
    "BLEU evaluation metric translation",
]

for q in test_qs:
    print(f"\nQuery: '{q}'")
    two   = retriever.retrieve(q, top_k=3)
    three = triple.retrieve(q, top_k=3)

    print("  2-way RRF (BM25+SBERT):")
    for r in two:
        print(f"    doc_{r['doc_id']}  BM25=#{r['bm25_rank']}  SBERT=#{r['sbert_rank']}"
              f"  RRF={r['rrf_score']:.6f}  {r['text'][:60]}...")

    print("  3-way RRF (BM25+SBERT+ColBERT):")
    for r in three:
        print(f"    doc_{r['doc_id']}  BM25=#{r['bm25_rank']}  SBERT=#{r['sbert_rank']}"
              f"  CB=#{r['colbert_rank']}  RRF={r['rrf_score']:.6f}  {r['text'][:55]}...")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


TripleHybridRetriever ready - 12 docs.

Query: 'how do GANs learn to generate realistic images?'
  2-way RRF (BM25+SBERT):
    doc_5  BM25=#1  SBERT=#4  RRF=0.032018  Diffusion models generate data by learning to reverse a grad...
    doc_3  BM25=#3  SBERT=#2  RRF=0.032002  Variational Autoencoders (VAEs) learn a latent distribution ...
    doc_4  BM25=#7  SBERT=#1  RRF=0.031319  Generative Adversarial Networks (GANs) pit a generator again...
  3-way RRF (BM25+SBERT+ColBERT):
    doc_5  BM25=#1  SBERT=#4  CB=#1  RRF=0.048412  Diffusion models generate data by learning to reverse a...
    doc_4  BM25=#7  SBERT=#1  CB=#2  RRF=0.047448  Generative Adversarial Networks (GANs) pit a generator ...
    doc_3  BM25=#3  SBERT=#2  CB=#5  RRF=0.047387  Variational Autoencoders (VAEs) learn a latent distribu...

Query: 'BLEU evaluation metric translation'
  2-way RRF (BM25+SBERT):
    doc_6  BM25=#1  SBERT=#1  RRF=0.032787  BLEU (Bilingual Evaluation Understudy) is an n-gram precisio...
    doc_0 

---

### ColBERT Observations
- ColBERT often **agrees with SBERT** on semantic queries but can diverge when fine-grained token-level matching matters.
- Adding ColBERT as a third retriever **stabilises the top results** — documents that rank highly across BM25, SBERT, and ColBERT receive a higher combined RRF score.
- ColBERT helps surface documents with **better term-level relevance**, especially for queries where exact wording is important.
- The trade-off: ColBERT is **slower at query time** because it performs late interaction over token embeddings instead of using a single dense vector.
- Overall, 3-way RRF (BM25 + SBERT + ColBERT) tends to produce **more robust and balanced rankings** compared to 2-way fusion.


---
## Summary

| Component | Choice | Library |
|---|---|---|
| Sparse retrieval | BM25 (`BM25Okapi`) | `rank-bm25` |
| Dense retrieval | SBERT (`all-MiniLM-L6-v2`) | `sentence-transformers` |
| Token-level retrieval | ColBERT MaxSim (word-level approx.) | `sentence-transformers` |
| Fusion | 2-way / 3-way RRF (k=60) | custom |
| Query expansion | Multi-Query (Gemini, 3 paraphrases) | `langchain-google-genai` |
| Re-ranking | Cross-Encoder (`ms-marco-MiniLM-L-6-v2`) | `sentence-transformers` |
| Generation | Gemini (`gemini-2.0-flash`) | `langchain-google-genai` |
| Chunking study | 50 / 100 / 200 word non-overlapping chunks | custom |

### Key Takeaways
1. **Multi-Query** broadens recall — different phrasings surface different documents, especially for ambiguous questions.
2. **3-way RRF** is more robust than 2-way because a document must rank well across three independent signals.
3. **100-word chunks** balance retrieval precision with contextual completeness.
4. **Cross-encoders re-rank accurately** but must be applied only to a small candidate set due to O(n) query-time cost.
